In [1]:
import unsloth  # noqa: F401  must be imported before transformers/trl
from unsloth import FastLanguageModel

import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

import pandas as pd
from tqdm import tqdm


# import warnings
# warnings.filterwarnings("ignore")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 05-14 15:12:26 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-14 15:12:26 [nixl_utils.py:34] NIXL is not available
WARNING 05-14 15:12:26 [nixl_utils.py:44] NIXL agent config is not available
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


/home/paradox/Desktop/ai/ai-env/lib/python3.12/site-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.0 is installed, but it is not compatible with the installed jaxlib version 0.10.0, so it will not be used.
  warnings.warn(


In [22]:
from datasets import load_dataset

ds = load_dataset("emirkaanozdemr/bash_command_data_6K")

In [23]:
ds.set_format("pandas")

In [29]:
import glob

In [33]:
for p in glob.glob("./datasets/cleaned/*.csv"):
    print(p)
    df = pd.read_csv(p)
   
    df = df.rename(columns={'input_text': 'prompt', 'bash_command': 'command'})
    df.to_csv(p, index=False)

./datasets/cleaned/nl2bash.csv
./datasets/cleaned/009.csv
./datasets/cleaned/004.csv
./datasets/cleaned/010.csv
./datasets/cleaned/000.csv
./datasets/cleaned/007.csv
./datasets/cleaned/001.csv
./datasets/cleaned/003.csv
./datasets/cleaned/002.csv
./datasets/cleaned/006.csv
./datasets/cleaned/008.csv
./datasets/cleaned/005.csv


In [51]:
# Slicing now returns a Pandas DataFrame
dictdf =  ds["train"].to_dict()
dictdf["prompt"] = [*map(lambda p: p.strip(), dictdf["prompt"])]
dictdf["completion"] = [*map(lambda p: p.strip(), dictdf["completion"])]
df = pd.DataFrame(dictdf)


In [52]:
df = df.rename(columns={"completion": "command"})

In [53]:
df.head()

,prompt,command
0,"List all files in the current directory, inclu...",ls -la
1,"Show the first 10 lines of a file named ""repor...",head -n 10 report.txt
2,"Count how many times the word ""error"" appears ...","grep -o ""error"" /var/log/syslog | wc -l"
3,Create a compressed tar archive named backup.t...,tar -czvf backup.tar.gz /home/user/data
4,Find all .log files larger than 5 MB in /var/l...,"find /var/log -type f -name ""*.log"" -size +5M ..."


In [55]:
import csv

In [58]:
df.to_csv("./datasets/cleaned/011.csv", index=False, quoting=csv.QUOTE_MINIMAL)

In [1]:
df.sample(n=20)

NameError: name 'df' is not defined

In [1]:
import pandas as pd

In [2]:
bdf = pd.read_csv("./results/baseline/009.csv")
pdf = pd.read_csv("./results/noshfinetunedV1/009.csv")

In [4]:
bdf.overall.mean(),  pdf.overall.mean()

(np.float64(3.64), np.float64(3.83))

In [14]:

xdf = pdf[pdf.overall < bdf.overall]
ydf = bdf[pdf.overall < bdf.overall]

In [15]:
xdf.shape

(13, 13)

In [36]:
i=5

In [37]:
xdf.iloc[i]

prompt                     merge two JSON files a.json and b.json
command                          join -j 1 -t $'\t' a.json b.json
expected                        jq -s '.[0] * .[1]' a.json b.json
exact_match                                                 False
formatting                                                   True
syntax_valid                                                 True
tool                                                        False
flags                                                           0
args                                                            0
constraints                                                     0
safety                                                       True
overall                                                         1
reasoning       Uses join which is not suitable for merging JS...
Name: 34, dtype: object

In [38]:

ydf.iloc[i]

prompt                     merge two JSON files a.json and b.json
command                                  cat a.json b.json | jq .
expected                        jq -s '.[0] * .[1]' a.json b.json
exact_match                                                 False
formatting                                                   True
syntax_valid                                                 True
tool                                                         True
flags                                                           1
args                                                            1
constraints                                                     1
safety                                                       True
overall                                                         3
reasoning       Uses cat and jq but does not merge JSON files ...
Name: 34, dtype: object